
# Notebook 1 — SVM Hierárquico de Condições e Subcategorias SCP

## Objetivo

Treinar classificadores SVM independentes para:

- MI — Infarto do miocárdio
- STTC — Alterações ST/T
- CD — Distúrbios de condução
- HYP — Hipertrofias
- FORM — Alterações morfológicas

Além dos cinco classificadores gerais, cada código SCP
pertencente às categorias terá seu próprio classificador
binário quando houver dados suficientes.

## Características

- ECG bruto da PTB-XL, preferencialmente a 500 Hz
- 12 derivações preservadas individualmente
- Características estatísticas e espectrais
- Picos R e características RR estimadas
- Idade exata
- Sexo
- Quatro grupos etários

## Estratégia de classificação

Os códigos são multirrótulo. Um mesmo ECG pode conter
mais de um código SCP.

As probabilidades dos códigos são independentes e não
precisam somar 100%.

**Importante:** uma probabilidade estimada não equivale
a uma certeza diagnóstica.

## Estratégia de validação

- Folds 1–8: treinamento dos SVMs
- Fold 9: escolha de C e calibração
- Fold 10: avaliação independente

A seleção e a calibração no mesmo fold 9 podem produzir
estimativas otimistas nesse conjunto. O fold 10 permanece
intocado até a avaliação final.



## 1. Instalação das dependências

Execute no terminal do VS Code:

```powershell
python -m pip install numpy pandas scipy scikit-learn wfdb joblib matplotlib
```

O notebook utiliza `scipy.integrate.trapezoid`, evitando
a dependência de `numpy.trapezoid`.

Configure a pasta BASE na célula seguinte.


In [1]:

from pathlib import Path
import ast
import json
import warnings

import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.integrate import trapezoid

import wfdb
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, roc_auc_score,
    average_precision_score, brier_score_loss,
    confusion_matrix
)

SEED = 42

BASE = Path(r"C:\Users\Matheus\Documents\PTB-XL")

METADATA_CSV = BASE / "ptbxl_database.csv"
SCP_CSV = BASE / "scp_statements.csv"

DESTINO = Path(
    r"C:\Users\Matheus\Documents\Modelos\modelos_01"
)

CACHE = BASE / "resultados_svm_reestruturados" / (
    "cache_condicoes_subcategorias_v1.pkl"
)

DESTINO.mkdir(parents=True, exist_ok=True)
CACHE.parent.mkdir(parents=True, exist_ok=True)

# 0 = processar todos os registros.
MAX_REGISTROS = 0

# Controle de custo computacional do SVM RBF.
# 0 = sem limite por classe.
MAX_TREINO_POR_CLASSE = 2500

MIN_POS_TREINO = 30
MIN_POS_CALIBRACAO = 15
MIN_POS_TESTE = 5

C_CANDIDATOS = [0.3, 1.0, 3.0]

LEADS = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

GRUPOS = ["MI", "STTC", "CD", "HYP", "FORM"]

assert METADATA_CSV.exists(), METADATA_CSV
assert SCP_CSV.exists(), SCP_CSV

print("BASE:", BASE)
print("DESTINO:", DESTINO)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)


BASE: C:\Users\Matheus\Documents\PTB-XL
DESTINO: C:\Users\Matheus\Documents\Modelos\modelos_01
NumPy: 1.26.4
SciPy: 1.13.0



## 2. Leitura e auditoria da PTB-XL

Os códigos individuais são extraídos de `scp_codes`.

O `scp_statements.csv` estabelece a relação entre
códigos e categorias.

- MI, STTC, CD, HYP: coluna `diagnostic_class`
- FORM: coluna `form`

Um código pode participar de mais de uma dimensão
descritiva. Não se força exclusividade artificial.

Registros com NORM e diagnóstico anormal simultâneo
serão identificados e excluídos conforme a regra
utilizada anteriormente no projeto.


In [2]:

df = pd.read_csv(METADATA_CSV)
scp = pd.read_csv(SCP_CSV, index_col=0)

scp.index = scp.index.map(str)

def parse_codes(value):
    try:
        obj = ast.literal_eval(value) if isinstance(value, str) else value
        if isinstance(obj, dict):
            return set(str(k).strip() for k in obj)
    except (SyntaxError, ValueError, TypeError):
        pass
    return set()

df["code_set"] = df["scp_codes"].map(parse_codes)

MAPA = {g: set() for g in GRUPOS}
MAPA_CODIGO = {}

for code, row in scp.iterrows():
    code = str(code)
    membership = set()

    diagnostic_class = row.get("diagnostic_class")

    if pd.notna(diagnostic_class):
        group = str(diagnostic_class).strip()
        if group in GRUPOS:
            membership.add(group)

    form = row.get("form")
    if pd.notna(form) and float(form) == 1:
        membership.add("FORM")

    MAPA_CODIGO[code] = membership

    for group in membership:
        MAPA[group].add(code)

def categorias_registro(codes):
    result = set()
    for code in codes:
        result.update(MAPA_CODIGO.get(code, set()))
    return result

df["categories"] = df["code_set"].map(categorias_registro)

DIAGNOSTICOS = {"MI", "STTC", "CD", "HYP"}

df["conflito_norm"] = df.apply(
    lambda r:
        ("NORM" in r["code_set"])
        and bool(r["categories"] & DIAGNOSTICOS),
    axis=1
)

print("Total:", len(df))
print("Conflitos NORM/anormal:", int(df.conflito_norm.sum()))

for grupo in GRUPOS:
    print(grupo, "códigos:", len(MAPA[grupo]))

# Pacientes não devem cruzar os folds existentes.
cross = (
    df.groupby("patient_id", dropna=True)["strat_fold"]
      .nunique()
)

if (cross > 1).any():
    raise ValueError(
        "Existem pacientes em folds diferentes: "
        "verifique a divisão antes do treinamento."
    )

metadata = df.loc[
    ~df["conflito_norm"] & df["code_set"].map(bool)
].copy()

if MAX_REGISTROS:
    metadata = (
        metadata.groupby("strat_fold", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(
                    len(x),
                    max(1, MAX_REGISTROS // 10)
                ),
                random_state=SEED
            )
        )
    )

display(
    pd.DataFrame([
        {
            "categoria": g,
            "quantidade_codigos": len(MAPA[g]),
            "codigos": ", ".join(sorted(MAPA[g]))
        }
        for g in GRUPOS
    ])
)


Total: 21799
Conflitos NORM/anormal: 445
MI códigos: 14
STTC códigos: 13
CD códigos: 11
HYP códigos: 5
FORM códigos: 19


,categoria,quantidade_codigos,codigos
0,MI,14,"ALMI, AMI, ASMI, ILMI, IMI, INJAL, INJAS, INJI..."
1,STTC,13,"ANEUR, DIG, EL, ISCAL, ISCAN, ISCAS, ISCIL, IS..."
2,CD,11,"1AVB, 2AVB, 3AVB, CLBBB, CRBBB, ILBBB, IRBBB, ..."
3,HYP,5,"LAO/LAE, LVH, RAO/RAE, RVH, SEHYP"
4,FORM,19,"ABQRS, DIG, HVOLT, INVT, LNGQT, LOWT, LPR, LVO..."



## 3. Extração de características dos sinais

São extraídas características por derivação, preservando
a identificação anatômica das 12 derivações.

Inclui:

- Desvio-padrão
- RMS
- Amplitude pico a pico
- MAD
- Variabilidade da primeira diferença
- Energia espectral em 1–5, 5–15 e 15–35 Hz
- Indicadores de qualidade e disponibilidade

A filtragem abaixo é apropriada para características
exploratórias, mas **não é utilizada para medir desvio
clínico de ST**.

A detecção de picos R é heurística e não equivale a
delineação clínica validada.


In [3]:

def clean_signal(x, fs):
    x = np.asarray(x, dtype=float)

    if len(x) < 3 * fs:
        return None

    good = np.isfinite(x)

    if good.mean() < 0.99:
        return None

    x = np.where(good, x, np.nanmedian(x[good]))

    hi = min(40.0, fs * 0.4)

    if hi <= 0.5:
        return None

    sos = signal.butter(
        3, [0.5, hi],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    return signal.sosfiltfilt(sos, x)


METRICAS_LEAD = [
    "std", "rms", "ptp", "mad",
    "deriv_mad", "energy_1_5",
    "energy_5_15", "energy_15_35"
]

def lead_features(raw, fs, lead):
    x = clean_signal(raw, fs)

    if x is None:
        out = {
            f"{lead}_{k}": np.nan
            for k in METRICAS_LEAD
        }
        out[f"{lead}_valid"] = 0
        return out

    out = {
        f"{lead}_valid": 1,
        f"{lead}_std": float(np.std(x)),
        f"{lead}_rms": float(np.sqrt(np.mean(x*x))),
        f"{lead}_ptp": float(np.ptp(x)),
        f"{lead}_mad": float(
            np.median(np.abs(x - np.median(x)))
        ),
        f"{lead}_deriv_mad": float(
            np.median(np.abs(np.diff(x)))
        )
    }

    freq, psd = signal.welch(
        x, fs=fs,
        nperseg=min(len(x), int(2*fs))
    )

    for a, b, name in [
        (1, 5, "energy_1_5"),
        (5, 15, "energy_5_15"),
        (15, 35, "energy_15_35")
    ]:
        mask = (freq >= a) & (freq < b)

        out[f"{lead}_{name}"] = (
            float(trapezoid(psd[mask], x=freq[mask]))
            if mask.sum() > 1
            else np.nan
        )

    return out


def rhythm_summary(x, fs):
    names = [
        "n_peaks", "hr_bpm", "rr_mean_ms",
        "rr_sd_ms", "rr_cv", "rmssd_ms",
        "rr_p10_ms", "rr_p90_ms"
    ]

    out = {k: np.nan for k in names}

    x = clean_signal(x, fs)

    if x is None:
        return out

    sos = signal.butter(
        2, [5, min(20, fs * 0.4)],
        btype="bandpass",
        fs=fs,
        output="sos"
    )

    y = signal.sosfiltfilt(sos, x)

    grad = np.gradient(y)

    width = max(3, int(0.08 * fs))

    energy = np.convolve(
        grad**2,
        np.ones(width) / width,
        mode="same"
    )

    peaks, _ = signal.find_peaks(
        energy,
        distance=max(1, int(0.28 * fs)),
        prominence=max(
            np.percentile(energy, 85) * 0.3,
            1e-12
        )
    )

    refined = []

    for p in peaks:
        left = max(0, p - int(0.075*fs))
        right = min(len(y), p + int(0.075*fs) + 1)

        if right > left:
            refined.append(
                left + int(np.argmax(np.abs(y[left:right])))
            )

    refined = np.unique(refined)

    if len(refined) > 1:
        keep = np.r_[
            True,
            np.diff(refined) > int(0.24 * fs)
        ]
        refined = refined[keep]

    rr = np.diff(refined) / fs

    rr = rr[(rr >= 0.28) & (rr <= 2.5)]

    if len(rr) < 4:
        return out

    out.update({
        "n_peaks": float(len(refined)),
        "hr_bpm": float(60 / np.mean(rr)),
        "rr_mean_ms": float(1000 * np.mean(rr)),
        "rr_sd_ms": float(1000 * np.std(rr)),
        "rr_cv": float(np.std(rr) / np.mean(rr)),
        "rmssd_ms": float(
            1000 * np.sqrt(np.mean(np.diff(rr)**2))
        ),
        "rr_p10_ms": float(1000 * np.quantile(rr, .1)),
        "rr_p90_ms": float(1000 * np.quantile(rr, .9))
    })

    return out


def extract_record(record):
    x = np.asarray(record.p_signal, dtype=float)
    fs = float(record.fs)

    names = list(record.sig_name)

    if x.ndim != 2 or x.shape[0] < int(5 * fs):
        return None

    out = {}

    for lead in LEADS:
        if lead in names:
            raw = x[:, names.index(lead)]
        else:
            raw = np.array([])

        out.update(lead_features(raw, fs, lead))

    reference = (
        "II" if "II" in names
        else ("I" if "I" in names else None)
    )

    if reference:
        out.update(
            rhythm_summary(x[:, names.index(reference)], fs)
        )
    else:
        out.update(
            rhythm_summary(np.array([]), fs)
        )

    return out

print("Extrator definido.")


Extrator definido.



## 4. Extração e cache

O cache guarda características derivadas do sinal,
sem modificar os arquivos WFDB originais.

O contexto de idade e sexo é aplicado posteriormente,
para permitir simulações com os mesmos sinais.

A primeira execução processará os sinais disponíveis.
As execuções seguintes poderão reutilizar o cache.


In [4]:

def build_cache(meta, path):
    if path.exists():
        cached = pd.read_pickle(path)

        if (
            isinstance(cached, pd.DataFrame)
            and "ecg_id" in cached.columns
            and not cached.empty
        ):
            print("Cache recuperado:", len(cached))
            return cached

    rows = []
    failures = []

    for i, (_, row) in enumerate(meta.iterrows(), 1):
        record = None

        for field in ["filename_hr", "filename_lr"]:
            filename = row.get(field)

            if not isinstance(filename, str):
                continue

            prefix = BASE / filename

            if not prefix.with_suffix(".hea").exists():
                continue

            try:
                record = wfdb.rdrecord(str(prefix))
                break
            except Exception:
                continue

        if record is None:
            failures.append(
                (row.ecg_id, "WFDB indisponível")
            )
            continue

        try:
            feat = extract_record(record)
        except Exception as exc:
            failures.append(
                (row.ecg_id, str(exc))
            )
            continue

        if feat is None:
            failures.append(
                (row.ecg_id, "sinal inválido")
            )
            continue

        feat["ecg_id"] = int(row.ecg_id)
        rows.append(feat)

        if i % 500 == 0:
            print(
                f"{i}/{len(meta)}; válidos={len(rows)}",
                flush=True
            )

    pd.DataFrame(
        failures,
        columns=["ecg_id", "motivo"]
    ).to_csv(
        path.with_suffix(".falhas.csv"),
        index=False
    )

    result = pd.DataFrame(rows)

    if result.empty:
        raise RuntimeError(
            "Nenhum sinal foi processado. "
            "Verifique records500/ e records100/."
        )

    result.to_pickle(path)

    print("Registros aproveitados:", len(result))

    return result


assert (
    (BASE / "records500").exists()
    or (BASE / "records100").exists()
), "Não foram encontradas pastas de sinais WFDB."

features_table = build_cache(metadata, CACHE)


Cache recuperado: 21354



## 5. Idade, sexo e faixas etárias

### Grupos da interface

- Pediátrico: 0–16
- Adulto jovem: 17–39
- Adulto: 40–59
- Idoso: 60+

A idade exata também permanece como característica.

Na PTB-XL:

- `sex = 0`: masculino
- `sex = 1`: feminino

As regras exclusivas para adultos utilizam idade
mínima de 18 anos, mesmo quando a interface agrupa
a idade de 17 anos em adulto jovem.

O modelo poderá aprender interações demográficas,
mas não se força uma mudança artificial nas probabilidades.

### Regras clínicas

O motor clínico original descreve limites para
FC, PR, QRS, QTc e ST.

Este notebook só extrai diretamente características
exploratórias de sinal e RR. Não possui delineação
clínica validada de PR, QRS, QTc e ponto J.

Consequentemente, não inventará essas medidas nem
transformará ausência de medição em normalidade.

As regras clínicas completas devem permanecer
separadas do SVM até existirem medidas válidas.


In [5]:

def age_group(age):
    if pd.isna(age) or age < 0 or age > 120:
        return "desconhecido"
    if age <= 16:
        return "pediatrico_0_16"
    if age <= 39:
        return "adulto_jovem_17_39"
    if age <= 59:
        return "adulto_40_59"
    return "idoso_60_mais"


AGE_GROUPS = [
    "pediatrico_0_16",
    "adulto_jovem_17_39",
    "adulto_40_59",
    "idoso_60_mais",
    "desconhecido"
]


def contextualize(frame):
    d = frame.copy()

    age = pd.to_numeric(
        d["age"], errors="coerce"
    )

    sex = pd.to_numeric(
        d["sex_ptbxl"], errors="coerce"
    )

    groups = age.map(age_group)

    d["age_missing"] = age.isna().astype(int)

    d["sex_masculino"] = (sex == 0).astype(int)
    d["sex_feminino"] = (sex == 1).astype(int)

    d["sex_desconhecido"] = (
        ~sex.isin([0, 1])
    ).astype(int)

    d["idade_maior_igual_18"] = (
        age >= 18
    ).astype(int)

    for group in AGE_GROUPS:
        d["faixa_" + group] = (
            groups == group
        ).astype(int)

    return d


merged = features_table.merge(
    metadata[
        [
            "ecg_id", "patient_id",
            "age", "sex", "strat_fold",
            "code_set", "categories"
        ]
    ],
    on="ecg_id",
    how="inner",
    validate="one_to_one"
)

merged = merged.rename(
    columns={"sex": "sex_ptbxl"}
)

merged = contextualize(merged)

IGNORAR = {
    "ecg_id", "patient_id", "strat_fold",
    "code_set", "categories"
}

FEATURE_COLS = [
    c for c in merged.columns
    if c not in IGNORAR
    and pd.api.types.is_numeric_dtype(merged[c])
]

print("ECGs:", len(merged))
print("Características:", len(FEATURE_COLS))
print("Folds:", merged.strat_fold.value_counts().sort_index())


ECGs: 21354
Características: 128
Folds: strat_fold
1     2133
2     2139
3     2142
4     2132
5     2128
6     2133
7     2128
8     2130
9     2142
10    2147
Name: count, dtype: int64



## 6. Alvos hierárquicos e multirrótulo

Esta etapa corrige a limitação do notebook anterior.

Serão construídos:

1. Cinco alvos gerais.
2. Alvos individuais para os códigos SCP de cada grupo.

O mapeamento real é obtido do `scp_statements.csv`.

Exemplo de nomenclatura:

- `MI`
- `MI__IMI`
- `MI__ASMI`
- `CD__CRBBB`

Os nomes representam tarefas distintas.

O mesmo código SCP pode pertencer a mais de uma
dimensão documentada. Essa informação é preservada.

Um modelo somente será treinado se houver exemplos
positivos e negativos suficientes nos folds exigidos.


In [6]:

LABELS = {}
LABEL_INFO = {}

# Cinco classificadores gerais
for group in GRUPOS:
    LABELS[group] = merged["categories"].map(
        lambda groups: int(group in groups)
    ).to_numpy(dtype=int)

    LABEL_INFO[group] = {
        "tipo": "categoria",
        "categoria": group,
        "codigo_scp": None
    }

# Subcategorias individuais
for group in GRUPOS:
    for code in sorted(MAPA[group]):

        name = f"{group}__{code}"

        LABELS[name] = merged["code_set"].map(
            lambda codes, code=code:
                int(code in codes)
        ).to_numpy(dtype=int)

        LABEL_INFO[name] = {
            "tipo": "subcategoria",
            "categoria": group,
            "codigo_scp": code
        }

audit = []

for label, y in LABELS.items():
    item = LABEL_INFO[label]

    audit.append({
        "modelo": label,
        "categoria": item["categoria"],
        "tipo": item["tipo"],
        "codigo_scp": item["codigo_scp"],
        "positivos": int(y.sum()),
        "negativos": int((1-y).sum())
    })

audit = pd.DataFrame(audit)

audit.to_csv(
    DESTINO / "auditoria_alvos.csv",
    index=False
)

display(audit)
print("Total de tarefas candidatas:", len(LABELS))


,modelo,categoria,tipo,codigo_scp,positivos,negativos
0,MI,MI,categoria,None,5468,15886
1,STTC,STTC,categoria,None,5202,16152
2,CD,CD,categoria,None,4483,16871
3,HYP,HYP,categoria,None,2644,18710
4,FORM,FORM,categoria,None,8888,12466
...,...,...,...,...,...,...
62,FORM__QWAVE,FORM,subcategoria,QWAVE,540,20814
63,FORM__STD_,FORM,subcategoria,STD_,1004,20350
64,FORM__STE_,FORM,subcategoria,STE_,28,21326
65,FORM__TAB_,FORM,subcategoria,TAB_,29,21325


Total de tarefas candidatas: 67



## 7. Treinamento SVM e calibração

### SVM

Cada alvo terá um classificador binário SVC com:

- Kernel RBF
- `class_weight="balanced"`
- Imputação mediana
- Padronização das variáveis
- Valores candidatos de C: 0,3; 1; 3

A pesquisa de C utilizará o fold 9.

### Probabilidades

O `decision_function()` do SVC não representa uma
probabilidade.

Para obter percentuais será utilizada calibração
sigmoidal de Platt:

1. Treinar SVM nos folds 1–8.
2. Obter escores no fold 9.
3. Ajustar uma regressão logística de calibração.
4. Aplicar SVM e calibrador no fold 10.

A calibração e a escolha de C compartilham o fold 9,
portanto o desempenho nesse fold é exploratório.

O desempenho definitivo será reportado no fold 10.

O calibrador aprende probabilidades relativas aos
rótulos anotados, não certezas clínicas.


In [7]:

def pipeline_svm(C):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
                keep_empty_features=True
            )
        ),
        ("scaler", StandardScaler()),
        (
            "svm",
            SVC(
                kernel="rbf",
                C=C,
                gamma="scale",
                class_weight="balanced",
                probability=False,
                random_state=SEED
            )
        )
    ])


def sample_train(y, indices, cap, seed):
    rng = np.random.default_rng(seed)
    selected = []

    for value in [0, 1]:
        ids = indices[y[indices] == value]

        if cap and len(ids) > cap:
            ids = rng.choice(
                ids, size=cap, replace=False
            )

        selected.extend(ids.tolist())

    return np.asarray(
        sorted(selected), dtype=int
    )


def metricas(y, probability, threshold=0.5):
    pred = (probability >= threshold).astype(int)

    cm = confusion_matrix(
        y, pred, labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    result = {
        "n": int(len(y)),
        "positivos": int(y.sum()),
        "accuracy": float(accuracy_score(y, pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y, pred)
        ),
        "precision": float(
            precision_score(y, pred, zero_division=0)
        ),
        "recall": float(
            recall_score(y, pred, zero_division=0)
        ),
        "f1": float(
            f1_score(y, pred, zero_division=0)
        ),
        "specificity": (
            float(tn / (tn + fp))
            if tn + fp else np.nan
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

    if len(np.unique(y)) == 2:
        result["roc_auc"] = float(
            roc_auc_score(y, probability)
        )
        result["average_precision"] = float(
            average_precision_score(y, probability)
        )
        result["brier_score"] = float(
            brier_score_loss(y, probability)
        )
    else:
        result.update({
            "roc_auc": np.nan,
            "average_precision": np.nan,
            "brier_score": np.nan
        })

    return result


def platt_fit(scores, targets):
    calibrator = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=SEED
    )

    calibrator.fit(
        np.asarray(scores).reshape(-1, 1),
        targets
    )

    return calibrator


def probability_from_scores(calibrator, scores):
    return calibrator.predict_proba(
        np.asarray(scores).reshape(-1, 1)
    )[:, 1]


print("Treinamento e calibração definidos.")


Treinamento e calibração definidos.



## 8. Treinar os classificadores gerais e específicos

Para cada alvo:

1. Confirmar quantidade de amostras.
2. Selecionar hiperparâmetro usando fold 9.
3. Ajustar o calibrador.
4. Avaliar no fold 10.
5. Salvar o pipeline e o calibrador.
6. Salvar a lista de características.
7. Registrar a categoria e o código SCP.

Modelos ignorados serão documentados separadamente.

**Atenção ao custo:** SVM RBF com muitos alvos pode
exigir bastante memória e processamento. A configuração
de amostragem do treinamento pode ser ajustada.


In [8]:

# ============================================================
# AUDITORIA COMPLETA DAS SUBCATEGORIAS SCP
# NÃO TREINA NOVAMENTE OS MODELOS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

DESTINO = Path(
    r"C:\Users\Matheus\Documents\Modelos\modelos_01"
)
DESTINO.mkdir(parents=True, exist_ok=True)

folds = merged["strat_fold"].to_numpy(dtype=int)

train_idx = np.flatnonzero(
    (folds >= 1) & (folds <= 8)
)
valid_idx = np.flatnonzero(folds == 9)
test_idx = np.flatnonzero(folds == 10)

registros = []

for label, y in LABELS.items():
    y = np.asarray(y, dtype=int)
    info = LABEL_INFO[label]

    treino = int(y[train_idx].sum())
    validacao = int(y[valid_idx].sum())
    teste = int(y[test_idx].sum())

    neg_treino = int(len(train_idx) - treino)
    neg_validacao = int(len(valid_idx) - validacao)
    neg_teste = int(len(test_idx) - teste)

    motivos = []

    if treino < MIN_POS_TREINO:
        motivos.append("Poucos positivos no treino")

    if neg_treino < 2:
        motivos.append("Poucos negativos no treino")

    if validacao < max(9, MIN_POS_CALIBRACAO):
        motivos.append("Poucos positivos na validação")

    if neg_validacao < 9:
        motivos.append("Poucos negativos na validação")

    if teste < MIN_POS_TESTE:
        motivos.append("Poucos positivos no teste")

    if neg_teste < 1:
        motivos.append("Sem negativos no teste")

    registros.append({
        "modelo": label,
        "grupo": info["categoria"],
        "tipo": info["tipo"],
        "codigo_scp": info["codigo_scp"],
        "total_positivos": int(y.sum()),
        "positivos_treino": treino,
        "positivos_validacao": validacao,
        "positivos_teste": teste,
        "negativos_treino": neg_treino,
        "negativos_validacao": neg_validacao,
        "negativos_teste": neg_teste,
        "elegivel_pelos_filtros_atuais": (
            len(motivos) == 0
        ),
        "motivo": (
            "; ".join(motivos)
            if motivos
            else "Atende aos critérios"
        )
    })

auditoria = pd.DataFrame(registros)

auditoria.to_csv(
    DESTINO / "auditoria_completa_subcategorias.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total de classificadores candidatos:", len(auditoria))

print(
    "Elegíveis pelos critérios atuais:",
    int(auditoria["elegivel_pelos_filtros_atuais"].sum())
)

print(
    "Ignorados pelos critérios atuais:",
    int((~auditoria["elegivel_pelos_filtros_atuais"]).sum())
)

print("\nAUDITORIA DE HYP")
display(
    auditoria[
        auditoria["grupo"] == "HYP"
    ]
)

print("\nRESUMO DE TODAS AS CATEGORIAS")
display(
    auditoria.groupby("grupo").agg(
        identificados=("modelo", "count"),
        elegiveis=(
            "elegivel_pelos_filtros_atuais",
            "sum"
        )
    )
)

print("\nArquivo exportado:")
print(
    DESTINO / "auditoria_completa_subcategorias.csv"
)


Total de classificadores candidatos: 67
Elegíveis pelos critérios atuais: 44
Ignorados pelos critérios atuais: 23

AUDITORIA DE HYP


,modelo,grupo,tipo,codigo_scp,total_positivos,positivos_treino,positivos_validacao,positivos_teste,negativos_treino,negativos_validacao,negativos_teste,elegivel_pelos_filtros_atuais,motivo
3,HYP,HYP,categoria,None,2644,2117,267,260,14948,1875,1887,True,Atende aos critérios
43,HYP__LAO/LAE,HYP,subcategoria,LAO/LAE,423,340,43,40,16725,2099,2107,True,Atende aos critérios
44,HYP__LVH,HYP,subcategoria,LVH,2131,1707,210,214,15358,1932,1933,True,Atende aos critérios
45,HYP__RAO/RAE,HYP,subcategoria,RAO/RAE,98,79,9,10,16986,2133,2137,False,Poucos positivos na validação
46,HYP__RVH,HYP,subcategoria,RVH,126,102,12,12,16963,2130,2135,False,Poucos positivos na validação
47,HYP__SEHYP,HYP,subcategoria,SEHYP,29,24,3,2,17041,2139,2145,False,Poucos positivos no treino; Poucos positivos n...



RESUMO DE TODAS AS CATEGORIAS


,identificados,elegiveis
grupo,,
CD,12,8
FORM,20,15
HYP,6,3
MI,15,9
STTC,14,9



Arquivo exportado:
C:\Users\Matheus\Documents\Modelos\modelos_01\auditoria_completa_subcategorias.csv


In [9]:

# ============================================================
# NOTEBOOK 1 — TREINAMENTO COMPLETO CORRIGIDO
# CATEGORIAS + SUBCATEGORIAS SCP
# ============================================================

from pathlib import Path
import re
import json
import hashlib
import zipfile
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
    brier_score_loss
)

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

SEED = globals().get("SEED", 42)

DESTINO = Path(
    r"C:\Users\Matheus\Documents\Modelos\modelos_01"
)

PASTAS = {
    "modelos": DESTINO,
    "graficos": DESTINO / "graficos",
    "relatorios": DESTINO / "relatorios",
    "previsoes": DESTINO / "previsoes"
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

C_CANDIDATOS = [0.3, 1.0, 3.0]

MIN_POS_TREINAR = 2
MIN_NEG_TREINAR = 2

BOOTSTRAP_N = 100

MAX_TREINO_POR_CLASSE = globals().get(
    "MAX_TREINO_POR_CLASSE", 2500
)

FAIXAS_ETARIAS = {
    "pediatrico": [0, 16],
    "adulto_jovem": [17, 39],
    "adulto": [40, 59],
    "idoso": [60, None]
}

SEX_ENCODING = {
    "masculino": 0,
    "feminino": 1
}

GRUPOS = ["MI", "STTC", "CD", "HYP", "FORM"]

# ============================================================
# 2. VERIFICAÇÕES
# ============================================================

for nome in [
    "merged",
    "FEATURE_COLS",
    "LABELS",
    "LABEL_INFO"
]:
    if nome not in globals():
        raise RuntimeError(
            f"Execute as células anteriores: {nome} ausente."
        )

assert "age" in FEATURE_COLS
assert "sex_ptbxl" in FEATURE_COLS

# Confirmar indicadores das quatro faixas.
INDICADORES_ETARIOS = [
    "faixa_pediatrico_0_16",
    "faixa_adulto_jovem_17_39",
    "faixa_adulto_40_59",
    "faixa_idoso_60_mais"
]

faltando_faixas = [
    c for c in INDICADORES_ETARIOS
    if c not in FEATURE_COLS
]

if faltando_faixas:
    raise RuntimeError(
        "Os indicadores de faixas etárias não estão "
        f"presentes em FEATURE_COLS: {faltando_faixas}. "
        "Execute a célula de contextualização antes "
        "de iniciar o treinamento."
    )

X = merged[FEATURE_COLS].to_numpy(dtype=float)
folds = merged.strat_fold.to_numpy(dtype=int)

TRAIN = np.flatnonzero(
    (folds >= 1) & (folds <= 8)
)
VALID = np.flatnonzero(folds == 9)
TEST = np.flatnonzero(folds == 10)

assert len(TRAIN) > 0
assert len(VALID) > 0
assert len(TEST) > 0

if "patient_id" in merged.columns:
    def pacientes(indices):
        return set(
            merged.iloc[indices].patient_id.dropna()
        )

    p1 = pacientes(TRAIN)
    p2 = pacientes(VALID)
    p3 = pacientes(TEST)

    assert not (p1 & p2), "Vazamento treino/validação"
    assert not (p1 & p3), "Vazamento treino/teste"
    assert not (p2 & p3), "Vazamento validação/teste"

print("Total de candidatos:", len(LABELS))
print("Treino:", len(TRAIN))
print("Validação:", len(VALID))
print("Teste:", len(TEST))

# ============================================================
# 3. NOMES SEGUROS PARA WINDOWS
# ============================================================

def nome_seguro(label):
    nome = re.sub(
        r'[<>:"/\\|?*\x00-\x1f]',
        "_",
        str(label)
    )
    return nome.rstrip(" .") or "sem_nome"


NOMES_ARQUIVOS = {}
usados = set()

for label in LABELS:
    nome = nome_seguro(label)

    if nome.casefold() in usados:
        sufixo = hashlib.sha256(
            str(label).encode("utf-8")
        ).hexdigest()[:8]

        nome = f"{nome}_{sufixo}"

    if nome.casefold() in usados:
        raise RuntimeError("Colisão de nomes de arquivos.")

    usados.add(nome.casefold())
    NOMES_ARQUIVOS[label] = nome

# ============================================================
# 4. PIPELINE SVM
# ============================================================

def criar_pipeline(C):
    try:
        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )
    except TypeError:
        imputer = SimpleImputer(
            strategy="median",
            add_indicator=True
        )

    return Pipeline([
        ("imputer", imputer),
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=C,
            gamma="scale",
            class_weight="balanced",
            probability=False,
            random_state=SEED
        ))
    ])


def amostrar_treino(y):
    rng = np.random.default_rng(SEED)

    selecionados = []

    for classe in [0, 1]:
        ids = TRAIN[y[TRAIN] == classe]

        if (
            MAX_TREINO_POR_CLASSE > 0
            and len(ids) > MAX_TREINO_POR_CLASSE
        ):
            ids = rng.choice(
                ids,
                size=MAX_TREINO_POR_CLASSE,
                replace=False
            )

        selecionados.extend(ids.tolist())

    return np.asarray(
        sorted(selecionados),
        dtype=int
    )

# ============================================================
# 5. CALIBRAÇÃO
# ============================================================

def calibrar_platt(scores, y):
    calibrador = LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=SEED
    )

    calibrador.fit(
        np.asarray(scores).reshape(-1, 1),
        y
    )

    return calibrador


def prob_platt(calibrador, scores):
    return calibrador.predict_proba(
        np.asarray(scores).reshape(-1, 1)
    )[:, 1]


def calibrar_cv(X_train, y_train, C=1.0):
    positivos = int((y_train == 1).sum())
    negativos = int((y_train == 0).sum())

    n_splits = min(
        3, positivos, negativos
    )

    if n_splits < 2:
        return None

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED
    )

    try:
        calibrated = CalibratedClassifierCV(
            estimator=criar_pipeline(C),
            method="sigmoid",
            cv=cv
        )
    except TypeError:
        calibrated = CalibratedClassifierCV(
            base_estimator=criar_pipeline(C),
            method="sigmoid",
            cv=cv
        )

    calibrated.fit(X_train, y_train)

    return calibrated

# ============================================================
# 6. SELEÇÃO DO LIMIAR
# ============================================================

def selecionar_limiar(y, prob):
    if len(np.unique(y)) < 2:
        return 0.5, np.nan

    p, r, thresholds = precision_recall_curve(
        y, prob
    )

    if len(thresholds) == 0:
        return 0.5, np.nan

    f1 = (
        2 * p[:-1] * r[:-1]
        / np.maximum(p[:-1] + r[:-1], 1e-12)
    )

    melhor = np.max(f1)
    candidatos = np.flatnonzero(
        np.isclose(f1, melhor)
    )

    idx = candidatos[
        np.argmax(p[:-1][candidatos])
    ]

    return float(thresholds[idx]), float(f1[idx])

# ============================================================
# 7. MÉTRICAS
# ============================================================

def calcular_ece(y, prob, bins=10):
    limites = np.linspace(0, 1, bins + 1)

    ids = np.digitize(
        prob, limites[1:-1], right=True
    )

    erro = 0.0

    for i in range(bins):
        mask = ids == i

        if not mask.any():
            continue

        erro += mask.mean() * abs(
            prob[mask].mean() - y[mask].mean()
        )

    return float(erro)


def calcular_metricas(y, prob, threshold):
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    duas_classes = len(np.unique(y)) == 2

    def divisao(num, den):
        return float(num / den) if den else np.nan

    result = {
        "n_teste": len(y),
        "positivos_teste": int(y.sum()),
        "negativos_teste": int((y == 0).sum()),

        "accuracy": float(accuracy_score(y, pred)),

        "balanced_accuracy": (
            float(balanced_accuracy_score(y, pred))
            if duas_classes else np.nan
        ),

        "precision": divisao(tp, tp + fp),
        "sensibilidade": divisao(tp, tp + fn),
        "recall": divisao(tp, tp + fn),
        "especificidade": divisao(tn, tn + fp),

        "f1": (
            float(f1_score(y, pred, zero_division=0))
            if (tp + fp + fn) else np.nan
        ),

        "auc_roc": (
            float(roc_auc_score(y, prob))
            if duas_classes else np.nan
        ),

        "auc_pr_ap": (
            float(average_precision_score(y, prob))
            if duas_classes else np.nan
        ),

        "brier_score": float(
            brier_score_loss(y, prob)
        ),
        "ece": calcular_ece(y, prob),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

    return result, pred


# ============================================================
# 8. INTERVALOS DE CONFIANÇA — BOOTSTRAP
# ============================================================

def bootstrap_ic(y, prob, threshold):
    if len(np.unique(y)) != 2:
        return {}

    rng = np.random.default_rng(SEED)

    idx0 = np.flatnonzero(y == 0)
    idx1 = np.flatnonzero(y == 1)

    nomes = [
        "accuracy",
        "balanced_accuracy",
        "precision",
        "sensibilidade",
        "especificidade",
        "f1",
        "auc_roc",
        "auc_pr_ap",
        "brier_score",
        "ece"
    ]

    series = {k: [] for k in nomes}

    for _ in range(BOOTSTRAP_N):
        ids = np.concatenate([
            rng.choice(idx0, len(idx0), replace=True),
            rng.choice(idx1, len(idx1), replace=True)
        ])

        m, _ = calcular_metricas(
            y[ids], prob[ids], threshold
        )

        for nome in nomes:
            series[nome].append(m[nome])

    result = {}

    for nome, values in series.items():
        arr = np.asarray(values, dtype=float)
        arr = arr[np.isfinite(arr)]

        if len(arr):
            result[f"{nome}_ic95_inf"] = float(
                np.percentile(arr, 2.5)
            )
            result[f"{nome}_ic95_sup"] = float(
                np.percentile(arr, 97.5)
            )

    return result

# ============================================================
# 9. GRÁFICOS
# ============================================================

def gerar_graficos(label, nome, y, prob, pred, metricas,
                   threshold):
    pasta = PASTAS["graficos"] / nome
    pasta.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(
        y, pred, labels=[0, 1]
    )

    pd.DataFrame(
        cm,
        index=["Real 0", "Real 1"],
        columns=["Previsto 0", "Previsto 1"]
    ).to_csv(
        pasta / "matriz_confusao.csv"
    )

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i, str(cm[i, j]),
                ha="center", va="center"
            )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")
    ax.set_title(f"Matriz de confusão — {label}")

    fig.tight_layout()
    fig.savefig(
        pasta / "matriz_confusao.png",
        dpi=180
    )
    plt.close(fig)

    # ROC / PR somente com duas classes no teste
    if len(np.unique(y)) == 2:
        fpr, tpr, _ = roc_curve(y, prob)

        pd.DataFrame({
            "fpr": fpr,
            "tpr": tpr
        }).to_csv(
            pasta / "curva_roc.csv",
            index=False
        )

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(
            fpr, tpr,
            label=f"AUC-ROC = {metricas['auc_roc']:.4f}"
        )
        ax.plot([0, 1], [0, 1], "--")
        ax.set_xlabel("Taxa de falsos positivos")
        ax.set_ylabel("Sensibilidade")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(
            pasta / "curva_roc.png", dpi=180
        )
        plt.close(fig)

        precision, recall, _ = (
            precision_recall_curve(y, prob)
        )

        pd.DataFrame({
            "recall": recall,
            "precision": precision
        }).to_csv(
            pasta / "curva_pr.csv",
            index=False
        )

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(
            recall, precision,
            label=f"AP = {metricas['auc_pr_ap']:.4f}"
        )
        ax.axhline(
            y.mean(), linestyle="--",
            label="Prevalência"
        )
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precisão")
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(
            pasta / "curva_pr.png", dpi=180
        )
        plt.close(fig)

        fraction, mean_pred = calibration_curve(
            y, prob, n_bins=10,
            strategy="quantile"
        )

        pd.DataFrame({
            "probabilidade_media": mean_pred,
            "frequencia_observada": fraction
        }).to_csv(
            pasta / "calibracao.csv",
            index=False
        )

        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot([0, 1], [0, 1], "--")
        ax.plot(mean_pred, fraction, "o-")
        ax.set_xlabel("Probabilidade estimada")
        ax.set_ylabel("Frequência observada")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(
            pasta / "calibracao.png", dpi=180
        )
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(
        prob[y == 0], bins=25,
        alpha=0.6, label="Ausente"
    )
    ax.hist(
        prob[y == 1], bins=25,
        alpha=0.6, label="Presente"
    )
    ax.axvline(
        threshold, linestyle="--",
        label=f"Limiar {threshold:.4f}"
    )
    ax.legend()
    ax.set_xlabel("Probabilidade")
    ax.set_ylabel("Quantidade")
    fig.tight_layout()
    fig.savefig(
        pasta / "distribuicao_probabilidades.png",
        dpi=180
    )
    plt.close(fig)

# ============================================================
# 10. AUDITORIA, RELATÓRIOS E EXPORTAÇÕES
# ============================================================

REPORT = []
IGNORADOS = []
AUDITORIA = []


def salvar_progresso():
    pd.DataFrame(REPORT).to_csv(
        DESTINO / "metricas_todos_modelos.csv",
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame(IGNORADOS).to_csv(
        DESTINO / "modelos_nao_treinados.csv",
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame(AUDITORIA).to_csv(
        DESTINO / "auditoria_todos_modelos.csv",
        index=False,
        encoding="utf-8-sig"
    )


def salvar_relatorio(item, nome):
    pasta = PASTAS["relatorios"] / nome
    pasta.mkdir(parents=True, exist_ok=True)

    pd.DataFrame([item]).to_csv(
        pasta / "metricas.csv",
        index=False,
        encoding="utf-8-sig"
    )

    with open(
        pasta / "metricas.json",
        "w", encoding="utf-8"
    ) as f:
        json.dump(
            item, f,
            ensure_ascii=False,
            indent=2,
            allow_nan=True
        )

    linhas = [
        f"{k}: {v}"
        for k, v in item.items()
    ]

    (pasta / "relatorio.txt").write_text(
        "\n".join(linhas),
        encoding="utf-8"
    )

# ============================================================
# 11. TREINAMENTO
# ============================================================

for numero, (label, y) in enumerate(
    LABELS.items(), 1
):
    y = np.asarray(y, dtype=int)

    info = LABEL_INFO[label]

    n_pos = int(y[TRAIN].sum())
    n_neg = int((y[TRAIN] == 0).sum())

    n_valid = int(y[VALID].sum())
    n_valid_neg = int((y[VALID] == 0).sum())

    n_test = int(y[TEST].sum())
    n_test_neg = int((y[TEST] == 0).sum())

    nome = NOMES_ARQUIVOS[label]
    filename = f"{nome}.joblib"

    entrada = {
        "modelo": label,
        "categoria": info["categoria"],
        "tipo": info["tipo"],
        "codigo_scp": info["codigo_scp"],
        "positivos_treino": n_pos,
        "negativos_treino": n_neg,
        "positivos_validacao": n_valid,
        "negativos_validacao": n_valid_neg,
        "positivos_teste": n_test,
        "negativos_teste": n_test_neg,
        "arquivo": filename,
        "status": "pendente",
        "motivo": ""
    }

    AUDITORIA.append(entrada)

    print(
        f"[{numero}/{len(LABELS)}] {label} | "
        f"Treino +{n_pos} / -{n_neg}"
    )

    # Critério baseado somente no treinamento
    if (
        n_pos < MIN_POS_TREINAR
        or n_neg < MIN_NEG_TREINAR
    ):
        entrada["status"] = "nao_treinavel"
        entrada["motivo"] = (
            "Menos de dois exemplos positivos "
            "ou negativos no treinamento"
        )
        IGNORADOS.append(entrada.copy())
        salvar_progresso()
        continue

    indices_treino = amostrar_treino(y)

    usar_fold9 = (
        n_valid >= 9
        and n_valid_neg >= 9
    )

    if usar_fold9:
        try:
            sel, resto = train_test_split(
                VALID,
                test_size=0.60,
                stratify=y[VALID],
                random_state=SEED
            )

            cal, lim = train_test_split(
                resto,
                test_size=0.50,
                stratify=y[resto],
                random_state=SEED + 1
            )

            usar_fold9 = all(
                len(np.unique(y[idx])) == 2
                for idx in [sel, cal, lim]
            )

        except ValueError:
            usar_fold9 = False

    model = None
    calibrator = None
    threshold = 0.5
    f1_limiar = np.nan
    C_final = 1.0
    ap_selecao = np.nan

    if usar_fold9:
        # Seleção de C no fold 9
        best = None

        for C in C_CANDIDATOS:
            candidate = criar_pipeline(C)

            candidate.fit(
                X[indices_treino],
                y[indices_treino]
            )

            scores = candidate.decision_function(
                X[sel]
            )

            ap = average_precision_score(
                y[sel], scores
            )

            if best is None or ap > best["ap"]:
                best = {
                    "model": candidate,
                    "C": C,
                    "ap": float(ap)
                }

        model = best["model"]
        C_final = best["C"]
        ap_selecao = best["ap"]

        scores_cal = model.decision_function(X[cal])

        calibrator = calibrar_platt(
            scores_cal, y[cal]
        )

        scores_lim = model.decision_function(X[lim])

        p_lim = prob_platt(
            calibrator, scores_lim
        )

        threshold, f1_limiar = selecionar_limiar(
            y[lim], p_lim
        )

        estrategia = "platt_fold9"

        scores_test = model.decision_function(
            X[TEST]
        )

        prob_test = prob_platt(
            calibrator, scores_test
        )

    else:
        # Calibração cruzada no treinamento
        model = calibrar_cv(
            X[indices_treino],
            y[indices_treino]
        )

        if model is None:
            entrada["status"] = "nao_treinavel"
            entrada["motivo"] = (
                "Calibração cruzada inviável"
            )
            IGNORADOS.append(entrada.copy())
            salvar_progresso()
            continue

        estrategia = "calibracao_cv_treino"

        if len(np.unique(y[VALID])) == 2:
            p_valid = model.predict_proba(
                X[VALID]
            )[:, 1]

            threshold, f1_limiar = selecionar_limiar(
                y[VALID], p_valid
            )

        prob_test = model.predict_proba(
            X[TEST]
        )[:, 1]

    # ========================================================
    # 12. AVALIAÇÃO INDEPENDENTE
    # ========================================================

    metricas, pred_test = calcular_metricas(
        y[TEST],
        prob_test,
        threshold
    )

    ic95 = bootstrap_ic(
        y[TEST],
        prob_test,
        threshold
    )

    # ========================================================
    # 13. EXPORTAR MODELO
    # ========================================================

    artifact = {
        "model": model,
        "calibrator": calibrator,
        "features": list(FEATURE_COLS),

        "label": str(label),
        "task": "condicoes_e_subcategorias",
        "categoria": info["categoria"],
        "tipo": info["tipo"],
        "codigo_scp": info["codigo_scp"],

        "threshold": float(threshold),
        "C": C_final,
        "calibration": estrategia,

        "probability_output": True,
        "probability_method": (
            "model.predict_proba"
            if calibrator is None
            else "calibrator.predict_proba"
        ),

        "sex_encoding": SEX_ENCODING,
        "age_groups": FAIXAS_ETARIAS,
        "train_folds": list(range(1, 9)),
        "validation_fold": 9,
        "test_fold": 10,

        "arquivo_modelo": filename,
        "note": (
            "Probabilidade exploratória de rótulo SCP. "
            "Não substitui diagnóstico médico."
        )
    }

    joblib.dump(
        artifact,
        DESTINO / filename,
        compress=3
    )

    # ========================================================
    # 14. EXPORTAR PREVISÕES
    # ========================================================

    dados_teste = merged.iloc[TEST]

    previsoes = pd.DataFrame({
        "ecg_id": dados_teste.ecg_id.to_numpy(),
        "patient_id": dados_teste.patient_id.to_numpy(),
        "idade": dados_teste.age.to_numpy(),
        "sexo": dados_teste.sex_ptbxl.to_numpy(),
        "y_true": y[TEST],
        "y_pred": pred_test,
        "probabilidade": prob_test,
        "percentual": prob_test * 100,
        "threshold": threshold
    })

    previsoes.to_csv(
        PASTAS["previsoes"] / f"{nome}_previsoes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # 15. RELATÓRIO E GRÁFICOS
    # ========================================================

    item = {
        "modelo": label,
        "categoria": info["categoria"],
        "tipo": info["tipo"],
        "codigo_scp": info["codigo_scp"],
        "arquivo": filename,
        "estrategia": estrategia,

        "C": C_final,
        "threshold": float(threshold),

        "train_pos": n_pos,
        "valid_pos": n_valid,
        "test_pos": n_test,

        "fold9_ap_selecao": ap_selecao,
        "fold9_f1_limiar": f1_limiar,

        **metricas,
        **ic95
    }

    gerar_graficos(
        label,
        nome,
        y[TEST],
        prob_test,
        pred_test,
        metricas,
        threshold
    )

    salvar_relatorio(item, nome)

    REPORT.append(item)

    entrada["status"] = (
        "treinado"
        if usar_fold9
        else "treinado_raro"
    )

    salvar_progresso()

    print(
        f"  {filename} | "
        f"F1={metricas['f1']} | "
        f"AUC={metricas['auc_roc']}"
    )

# ============================================================
# 16. INVENTÁRIO COMPLETO
# ============================================================

treinados = {
    item["modelo"]: item
    for item in REPORT
}

INVENTARIO = {
    "versao": "5.0",
    "tipo": "condicoes_e_subcategorias",
    "feature_cols": list(FEATURE_COLS),
    "faixas_etarias": FAIXAS_ETARIAS,
    "sexo": SEX_ENCODING,
    "modelos": {},
    "categorias": {}
}

for grupo in GRUPOS:
    INVENTARIO["categorias"][grupo] = {
        "modelo_geral": None,
        "subcategorias": []
    }

for item in AUDITORIA:
    label = item["modelo"]
    grupo = item["categoria"]

    registro = {
        "label": label,
        "categoria": grupo,
        "tipo": item["tipo"],
        "codigo_scp": item["codigo_scp"],
        "status": item["status"],
        "motivo": item["motivo"],
        "arquivo": (
            treinados[label]["arquivo"]
            if label in treinados else None
        )
    }

    INVENTARIO["modelos"][label] = registro

    if item["tipo"] == "categoria":
        INVENTARIO["categorias"][grupo][
            "modelo_geral"
        ] = registro
    else:
        INVENTARIO["categorias"][grupo][
            "subcategorias"
        ].append(registro)

with open(
    DESTINO / "inventario_modelos.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        INVENTARIO, f,
        indent=2, ensure_ascii=False
    )

salvar_progresso()

# ============================================================
# 17. RESUMO
# ============================================================

report_df = pd.DataFrame(REPORT)
ignored_df = pd.DataFrame(IGNORADOS)
auditoria_df = pd.DataFrame(AUDITORIA)

print("\nTREINAMENTO FINALIZADO")
print("Candidatos:", len(LABELS))
print("Treinados:", len(REPORT))
print("Não treináveis:", len(IGNORADOS))

display(
    auditoria_df.groupby(
        ["categoria", "status"]
    ).size().unstack(fill_value=0)
)

if not report_df.empty:
    display(
        report_df[
            [
                "modelo", "estrategia",
                "accuracy", "sensibilidade",
                "especificidade", "precision",
                "f1", "auc_roc", "auc_pr_ap",
                "brier_score", "ece"
            ]
        ]
    )

# ============================================================
# 18. ZIP DOS RESULTADOS
# ============================================================

zip_path = DESTINO / "resultados_completos_svm.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_STORED
) as z:

    for arquivo in DESTINO.rglob("*"):
        if (
            arquivo.is_file()
            and arquivo.resolve() != zip_path.resolve()
        ):
            z.write(
                arquivo,
                arcname=arquivo.relative_to(DESTINO)
            )

print("Saída:", DESTINO.resolve())
print("ZIP:", zip_path.resolve())


Total de candidatos: 67
Treino: 17065
Validação: 2142
Teste: 2147
[1/67] MI | Treino +4378 / -12687
  MI.joblib | F1=0.5937007874015748 | AUC=0.8159423919849719
[2/67] STTC | Treino +4162 / -12903
  STTC.joblib | F1=0.6177325581395349 | AUC=0.8480153314900737
[3/67] CD | Treino +3575 / -13490
  CD.joblib | F1=0.5882352941176471 | AUC=0.8421934884745844
[4/67] HYP | Treino +2117 / -14948
  HYP.joblib | F1=0.5553772070626003 | AUC=0.8772165830989359
[5/67] FORM | Treino +7121 / -9944
  FORM.joblib | F1=0.649763644177052 | AUC=0.745485981863007
[6/67] MI__ALMI | Treino +232 / -16833
  MI__ALMI.joblib | F1=0.17647058823529413 | AUC=0.884958071278826
[7/67] MI__AMI | Treino +282 / -16783
  MI__AMI.joblib | F1=0.21212121212121213 | AUC=0.7831980519480519
[8/67] MI__ASMI | Treino +1887 / -15178
  MI__ASMI.joblib | F1=0.5611285266457681 | AUC=0.919498170412964
[9/67] MI__ILMI | Treino +383 / -16682
  MI__ILMI.joblib | F1=0.35514018691588783 | AUC=0.8926770684452914
[10/67] MI__IMI | Treino +21

status,treinado,treinado_raro
categoria,,
CD,8,4
FORM,16,4
HYP,5,1
MI,9,6
STTC,13,1


,modelo,estrategia,accuracy,sensibilidade,especificidade,precision,f1,auc_roc,auc_pr_ap,brier_score,ece
0,MI,platt_fold9,0.759665,0.685455,0.785222,0.523611,0.593701,0.815942,0.613507,0.142096,0.016200
1,STTC,platt_fold9,0.755007,0.822050,0.733742,0.494761,0.617733,0.848015,0.643059,0.126282,0.021882
2,CD,platt_fold9,0.801118,0.679287,0.833333,0.518707,0.588235,0.842193,0.618907,0.116996,0.016873
3,HYP,platt_fold9,0.870983,0.665385,0.899311,0.476584,0.555377,0.877217,0.598286,0.071702,0.011964
4,FORM,platt_fold9,0.620401,0.865979,0.452119,0.519945,0.649764,0.745486,0.648211,0.200262,0.026712
...,...,...,...,...,...,...,...,...,...,...,...
62,FORM__QWAVE,platt_fold9,0.861667,0.301887,0.875836,0.057971,0.097264,0.718666,0.061220,0.023853,0.002657
63,FORM__STD_,platt_fold9,0.927806,0.232323,0.961426,0.225490,0.228856,0.791341,0.171366,0.041252,0.008336
64,FORM__STE_,calibracao_cv_treino,0.986493,0.000000,0.987873,0.000000,0.000000,0.776275,0.004128,0.001455,0.007609
65,FORM__TAB_,calibracao_cv_treino,0.857476,0.000000,0.858675,0.000000,0.000000,0.647233,0.002541,0.001454,0.007545


Saída: C:\Users\Matheus\Documents\Modelos\modelos_01
ZIP: C:\Users\Matheus\Documents\Modelos\modelos_01\resultados_completos_svm.zip



## 9. Inventário de modelos para o visualizador

O inventário identifica:

- Categoria principal
- Subcategoria SCP
- Nome do arquivo `.joblib`
- Características esperadas
- Tipo de probabilidade
- Limiar de classificação

O visualizador deverá carregar esse inventário para
apresentar as subcategorias da condição selecionada.

Os ritmos continuam pertencendo a outro conjunto de
modelos e não são tratados como subcategorias de MI,
STTC, CD, HYP ou FORM.


In [10]:

INVENTARIO = {
    "versao": "1.0",
    "tipo": "condicoes_hierarquicas_multirrotulo",
    "categorias": {},
    "feature_cols": list(FEATURE_COLS),
    "folds": {
        "treino": list(range(1, 9)),
        "calibracao": 9,
        "teste": 10
    },
    "probabilidades": (
        "Calibradas por regressão logística no fold 9"
    ),
    "observacao": (
        "Probabilidades de rótulos SCP; "
        "não representam certeza diagnóstica."
    )
}

trained = {
    item["modelo"]: item
    for item in REPORT
}

for group in GRUPOS:

    general = (
        f"{group}.joblib"
        if group in trained
        else None
    )

    children = []

    for code in sorted(MAPA[group]):
        label = f"{group}__{code}"

        if label not in trained:
            continue

        description = None

        if code in scp.index:
            description = scp.loc[code].get(
                "description"
            )

        children.append({
            "codigo": code,
            "descricao": (
                str(description)
                if pd.notna(description)
                else code
            ),
            "modelo": f"{label}.joblib"
        })

    INVENTARIO["categorias"][group] = {
        "modelo_geral": general,
        "subcategorias": children
    }

with open(
    DESTINO / "inventario_modelos.json",
    "w", encoding="utf-8"
) as f:
    json.dump(
        INVENTARIO, f,
        ensure_ascii=False, indent=2
    )

print("Inventário exportado.")
print(
    "Total de modelos disponíveis:",
    len(REPORT)
)


Inventário exportado.
Total de modelos disponíveis: 67



## 10. Exemplo de inferência contextual

O mesmo ECG pode ser avaliado usando diferentes
idades e sexos.

A aplicação deve:

1. Preservar as características do sinal.
2. Atualizar a idade e o sexo selecionados.
3. Recalcular os indicadores de faixa etária.
4. Reorganizar o vetor na ordem de treinamento.
5. Executar o modelo e o calibrador.
6. Retornar a probabilidade estimada.

Uma nova inferência é obrigatória quando muda o
contexto, mas uma mudança numérica na probabilidade
não é garantida.

As probabilidades de cada subcategoria são independentes.


In [11]:

def predict_artifact(artifact, frame):
    cols = artifact["features"]

    for col in cols:
        if col not in frame.columns:
            raise ValueError(
                f"Característica ausente: {col}"
            )

    x = frame[cols].to_numpy(dtype=float)

    scores = artifact["model"].decision_function(x)

    probabilities = probability_from_scores(
        artifact["calibrator"],
        scores
    )

    return probabilities


def simular_contexto(
    ecg_id,
    idade,
    sexo,
    categoria="MI"
):
    # sexo: 0 masculino, 1 feminino
    if sexo not in (0, 1):
        raise ValueError("Sexo deve ser 0 ou 1.")

    base_row = merged.loc[
        merged.ecg_id == ecg_id
    ].copy()

    if len(base_row) != 1:
        raise ValueError("ECG não encontrado ou duplicado.")

    # Mantém as características reais do ECG.
    base_row["age"] = idade
    base_row["sex_ptbxl"] = sexo

    # Recria indicadores contextuais.
    base_row = contextualize(base_row)

    results = []

    config = INVENTARIO["categorias"][categoria]

    models = []

    if config["modelo_geral"]:
        models.append(config["modelo_geral"])

    models.extend(
        item["modelo"]
        for item in config["subcategorias"]
    )

    for filename in models:
        artifact = joblib.load(
            DESTINO / filename
        )

        p = predict_artifact(
            artifact, base_row
        )[0]

        results.append({
            "modelo": artifact["label"],
            "categoria": artifact["categoria"],
            "codigo_scp": artifact["codigo_scp"],
            "idade": idade,
            "sexo": sexo,
            "probabilidade": float(p),
            "percentual": round(100 * float(p), 2)
        })

    return pd.DataFrame(results)


# Exemplo executável após o treinamento:
#
# ecg_exemplo = int(merged.iloc[0].ecg_id)
#
# display(simular_contexto(
#     ecg_exemplo, idade=25, sexo=0, categoria="MI"
# ))
#
# display(simular_contexto(
#     ecg_exemplo, idade=65, sexo=1, categoria="MI"
# ))



## 11. Interpretação dos resultados e limitações

### O que o notebook entrega

- Um classificador geral por categoria elegível.
- Classificadores individuais por código SCP elegível.
- Probabilidades calibradas de rótulo.
- Características das 12 derivações.
- Características de RR estimadas.
- Idade e sexo.
- Indicadores das quatro faixas etárias.
- Avaliação independente no fold 10.
- Inventário para integração no visualizador.

### O que não deve ser afirmado

- Que cada probabilidade representa certeza clínica.
- Que uma subcategoria foi diagnosticada definitivamente.
- Que todas as subcategorias terão dados suficientes.
- Que mudar idade/sexo sempre mudará a pontuação.
- Que PR, QRS, QTc e ST foram medidos clinicamente
  quando o extrator não fornece essas medidas.

### Integração com o visualizador

Ao selecionar MI, apresentar apenas as probabilidades
dos códigos mapeados para MI.

O mesmo deve ocorrer com STTC, CD, HYP e FORM.

A seleção de uma categoria altera a organização da
exibição, não o sinal nem os rótulos originais.

Os modelos de ritmo devem continuar complementares
e separados.

### Próxima validação necessária

Comparar os modelos com e sem características
demográficas, verificar a calibração por faixa etária
e sexo e avaliar desempenho em pacientes independentes
e, idealmente, em bases externas.
